In [ ]:
# standard libraries
import os
import sys
import copy
import math
import re
import json
from pathlib import Path
from datetime import datetime
import platform

# data manipulation
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# optimization
from scipy.optimize import minimize

# sklearn
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import clone
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.preprocessing import LabelEncoder

# boosting & tuning
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
import joblib
import shap

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# display
from IPython.display import display, HTML

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 250)

SEED = 42

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
ARTIFACT_DIR = REPO_ROOT / "models"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


# 6. Modeling

In [ ]:
X_train_full_s    = pd.read_parquet(PROCESSED_DIR / "X_train_full_s.parquet")
X_valid_full_s    = pd.read_parquet(PROCESSED_DIR / "X_valid_full_s.parquet")
X_test_full_s     = pd.read_parquet(PROCESSED_DIR / "X_test_full_s.parquet")

X_train_reduced_s = pd.read_parquet(PROCESSED_DIR / "X_train_reduced_s.parquet")
X_valid_reduced_s = pd.read_parquet(PROCESSED_DIR / "X_valid_reduced_s.parquet")
X_test_reduced_s  = pd.read_parquet(PROCESSED_DIR / "X_test_reduced_s.parquet")

y_train = pd.read_parquet(PROCESSED_DIR / "y_train.parquet")["target"]
y_valid = pd.read_parquet(PROCESSED_DIR / "y_valid.parquet")["target"]
y_test  = pd.read_parquet(PROCESSED_DIR / "y_test.parquet")["target"]

In [ ]:
# load encoding artifacts produced by notebook 04
enc = joblib.load(PROCESSED_DIR / "encoding_artifacts.joblib")
sanit_map        = enc["sanit_map"]
label_encoders   = enc["label_encoders"]
categorical_full = enc["categorical_full"]

Objective: predict log-transformed property price

Approach:
 - linear models as baseline
 - tree-based ensembles
 -  blending to test performance gains

Setup:
 - train / validation / test split
 - 5-fold cross-validation for tuning
 - metric: RMSE (primary), MAE, R²

Data:
 - linear models: feature selection + scaled + OHE (reduced dataset)
 - tree models: full dataset with label encoding

All models are tuned using Optuna with cross-validation


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

linear_models = ["LinearRegression", "Ridge", "Lasso", "ElasticNet"]
tree_models   = ["RandomForest", "GradientBoosting", "XGBoost", "LightGBM", "CatBoost"]
model_registry = {
    "LinearRegression": LinearRegression,
    "Ridge": Ridge,
    "Lasso": Lasso,
    "ElasticNet": ElasticNet,
    "RandomForest": RandomForestRegressor,
    "GradientBoosting": GradientBoostingRegressor,
    "XGBoost": XGBRegressor,
    "LightGBM": LGBMRegressor,
    "CatBoost": CatBoostRegressor
}

def objective(trial, model_name, X, y):

    if model_name == "LinearRegression":
        model = LinearRegression()

    elif model_name == "Ridge":
        model = Ridge(alpha=trial.suggest_float("alpha", 1e-3, 5, log=True))

    elif model_name == "Lasso":
        model = Lasso(
            alpha=trial.suggest_float("alpha", 1e-5, 0.5, log=True),
            max_iter=5000
        )

    elif model_name == "ElasticNet":
        model = ElasticNet(
            alpha=trial.suggest_float("alpha", 1e-5, 0.5, log=True),
            l1_ratio=trial.suggest_categorical("l1_ratio", [0.1,0.3,0.5,0.7,0.9]),
            max_iter=5000
        )

    elif model_name == "RandomForest":
        model = RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 600, step=100),
            max_depth=trial.suggest_int("max_depth", 5, 20),
            random_state=SEED,
            n_jobs=-1
        )

    elif model_name == "GradientBoosting":
        model = GradientBoostingRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 600, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
            max_depth=trial.suggest_int("max_depth", 2, 5),
            random_state=SEED
        )

    elif model_name == "XGBoost":
        model = XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 800, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
            max_depth=trial.suggest_int("max_depth", 3, 8),
            subsample=trial.suggest_float("subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.7, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 5, log=True),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 5, log=True),
            random_state=SEED,
            tree_method="hist",
            verbosity=0
        )

    elif model_name == "LightGBM":
        model = LGBMRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 800, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
            num_leaves=trial.suggest_int("num_leaves", 31, 127),
            min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 20, 80),
            random_state=SEED,
            verbose=-1
        )

    elif model_name == "CatBoost":
        model = CatBoostRegressor(
            iterations=trial.suggest_int("iterations", 500, 1200, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15),
            depth=trial.suggest_int("depth", 4, 8),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1, 8),
            loss_function="RMSE",
            verbose=0,
            random_seed=SEED
        )

    scores = cross_val_score(
        model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1
    )

    return -scores.mean()


note on CatBoost: it supports native categorical encoding via the `cat_features` parameter, which uses ordered target statistics per category and handles high-cardinality features more rigorously than label encoding. for this project label encoding is used across all tree models for consistency, to keep sklearn-based visualization tools (SHAP beeswarm, partial dependence plots) working without workarounds. the performance difference on this dataset is negligible.

In [ ]:
# training and leaderboard
results = []
best_models = {}
best_params_store = {}

for group in [linear_models, tree_models]:

    for model_name in group:

        print(f"\nTuning {model_name}...")

        Xtr  = X_train_reduced_s if model_name in linear_models else X_train_full_s
        Xval = X_valid_reduced_s if model_name in linear_models else X_valid_full_s

        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.TPESampler(seed=SEED)
        )

        study.optimize(
            lambda trial: objective(trial, model_name, Xtr, y_train),
            n_trials=40 if model_name in linear_models else 60, # fewer trials for linear (faster), more for tree models
            show_progress_bar=False
        )

        model = model_registry[model_name](**study.best_params)
        model.fit(Xtr, y_train)

        val_preds = model.predict(Xval)

        results.append({
            "Model": model_name,
            "RMSE": np.sqrt(mean_squared_error(y_valid, val_preds)),
            "MAE": mean_absolute_error(y_valid, val_preds),
            "R2": r2_score(y_valid, val_preds)
        })

        best_models[model_name] = model

leaderboard = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)



# ensemble: weighted blend (top 3)

top_models = leaderboard["Model"].iloc[:3].tolist()

def get_val_features(model_name):
    return X_valid_reduced_s if model_name in linear_models else X_valid_full_s

val_preds = np.column_stack([
    best_models[m].predict(get_val_features(m))
    for m in top_models
])

def blend_loss(weights):
    blended = np.dot(val_preds, weights)
    return mean_squared_error(y_valid, blended)

constraints = {"type": "eq", "fun": lambda w: 1 - np.sum(w)}
bounds = [(0, 1)] * len(top_models)
init = np.ones(len(top_models)) / len(top_models)

result = minimize(blend_loss, init, bounds=bounds, constraints=constraints)

blend_weights = result.x
blend_preds = np.dot(val_preds, blend_weights)

results.append({
    "Model": "WeightedBlend",
    "RMSE": np.sqrt(mean_squared_error(y_valid, blend_preds)),
    "MAE": mean_absolute_error(y_valid, blend_preds),
    "R2": r2_score(y_valid, blend_preds)
})

leaderboard = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


print("\n Leaderboard")
print(leaderboard)

Models are ranked based on validation RMSE.
Tree-based ensembles significantly outperform linear models, which indicates strong non-linear relationships in the dataset.

Weighted blend outperforms all individual models. CatBoost is the strongest single model, followed closely by XGBoost.

Linear models perform well but underfit relative to boosting methods.

In [ ]:
print("blend weights:")
for model, w in zip(top_models, blend_weights):
    print(f"  {model}: {w:.3f}")

In [ ]:
display(leaderboard)
leaderboard.to_csv(PROCESSED_DIR / "leaderboard.csv", index=False)

# 7. Final test evaluation, interpretation & deployment

CatBoost is selected as the final model. the weighted blend had slightly better validation RMSE (0.169 vs 0.170) but the difference is small enough that a single model is simpler to deploy and maintain.

In [ ]:
final_model_name = "CatBoost"
print("Selected final model:", final_model_name)

Selected final model: CatBoost


In [ ]:
# refit on train+validation
X_trv_full = pd.concat([X_train_full_s, X_valid_full_s])
y_trv = pd.concat([y_train, y_valid])

final_model = clone(best_models[final_model_name])
final_model.fit(X_trv_full, y_trv)

# test evaluation
test_preds = final_model.predict(X_test_full_s)

rmse_test = np.sqrt(mean_squared_error(y_test, test_preds))
mae_test  = mean_absolute_error(y_test, test_preds)
r2_test   = r2_score(y_test, test_preds)

print(f"""
TEST PERFORMANCE
RMSE: {rmse_test:.5f}
MAE : {mae_test:.5f}
R2  : {r2_test:.5f}
""")

In [ ]:
# bootstrap 95% confidence interval for test RMSE
from sklearn.utils import resample

boot_rmse = []
for _ in range(1000):
    idx_b = resample(range(len(y_test)), random_state=None)
    boot_rmse.append(np.sqrt(mean_squared_error(
        y_test.values[idx_b], test_preds[idx_b]
    )))

ci_lo, ci_hi = np.percentile(boot_rmse, [2.5, 97.5])
print(f"RMSE 95% CI: ({ci_lo:.4f}, {ci_hi:.4f})")

In [ ]:
residuals = y_test.values - test_preds

plt.figure(figsize=(6,4))
plt.scatter(test_preds, residuals, s=12, alpha=0.5)
plt.axhline(0, ls="--")
plt.xlabel("Predicted (log)")
plt.ylabel("Residual (log)")
plt.title("Residuals vs Predicted (log scale)")
plt.grid(True)
plt.gca().set_axisbelow(True)
plt.tight_layout()
plt.show()

residuals scattered randomly around zero. no systematic over- or underprediction, no visible fan shape. a few outliers at both ends are probably atypical properties (very high-end listings or unusual layouts) that the model hasn't seen enough of during training.

In [ ]:
y_price_tst    = np.exp(y_test)
yhat_price_tst = np.exp(test_preds)

abs_pct_err = np.abs(yhat_price_tst - y_price_tst) / np.clip(y_price_tst, 1e-6, None)

plt.figure(figsize=(6,4))
plt.hist(abs_pct_err, bins=40)
plt.title("Absolute Percentage Error Distribution (Test)")
plt.grid(True, axis='y')
plt.gca().set_axisbelow(True)
plt.tight_layout()
plt.show()

print("Median absolute percentage error:", float(np.median(abs_pct_err)))

median absolute percentage error of ~7.8%. for a typical 5M CZK apartment that's roughly 390 000 CZK. the right tail has harder cases: unusual layouts, rare locations, or extreme prices. most predictions fall within ±15% of the actual price.

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(
    final_model,
    X_test_full_s,
    y_test,
    scoring="neg_mean_squared_error",
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test_full_s.columns,
    "importance": pi.importances_mean,
    "std": pi.importances_std
}).sort_values("importance", ascending=False)

topk = importance_df.head(20)

plt.figure(figsize=(8, 6))
plt.barh(topk["feature"][::-1], topk["importance"][::-1])
plt.xlabel("Decrease in performance")
plt.title("Permutation Importance (Top 20 Features)")
plt.grid(True, axis='x')
plt.gca().set_axisbelow(True)
plt.tight_layout()
plt.show()

size features dominate (square_meters, total_area), confirming apartment size is the strongest price driver. location (district, cadastral_area) and layout rank next, consistent with EDA. floor number, building characteristics and binary presence features (cellar, parking) contribute less but are non-zero.

In [ ]:
sample_size = min(1000, len(X_test_full_s))
X_sample = X_test_full_s.sample(sample_size, random_state=SEED)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer(X_sample)

shap.plots.beeswarm(shap_values, max_display=15)

size features show the expected SHAP pattern. larger apartments push predictions up consistently. layout shows distinct clusters per category, not a continuous gradient, which makes sense since it's label-encoded. district values scatter in both directions because the label numbers have no ordinal meaning. each district is a separate tree condition. building type forms a few sharp bands: panel buildings sit clearly on the negative side, brick varies more depending on location.

In [ ]:
# visualize marginal effect of top features
from sklearn.inspection import PartialDependenceDisplay

key_feats = topk["feature"].head(3).tolist()

fig, axes = plt.subplots(len(key_feats), 1, figsize=(6, 4*len(key_feats)))

if len(key_feats) == 1:
    axes = [axes]

for ax, feat in zip(axes, key_feats):
    disp = PartialDependenceDisplay.from_estimator(
        final_model,
        X_test_full_s,
        [feat],
        grid_resolution=30,
        ax=ax
    )
    for a in disp.axes_.ravel():
        a.grid(True)
        a.set_axisbelow(True)

plt.tight_layout()
plt.show()

most features show monotonic effects. larger area, newer buildings, higher floors all move price predictably. district is the exception: the non-monotonic pattern is an artifact of label encoding. the 11 labels map alphabetically: 0 = Missing, 1 = Praha 1 (most expensive), 2 = Praha 10 (cheapest), 3 = Praha 2, continuing to 10 = Praha 9. the model learned each district's price level correctly. the irregular shape just reflects that encoded integers carry no ordinal meaning.

In [ ]:
# save artifacts and metadata

artifact_dir = ARTIFACT_DIR
artifact_dir.mkdir(parents=True, exist_ok=True)

# define metadata elements
TARGET_TRANSFORM = "log"
feature_names = X_train_full_s.columns.tolist()
# use median for continuous features, mode for label-encoded categoricals
inv_sanit = {v: k for k, v in sanit_map.items()}
feature_defaults = {}
for col in X_train_full_s.columns:
    orig = inv_sanit.get(col, col)
    if orig in label_encoders:
        feature_defaults[col] = int(X_train_full_s[col].mode()[0])
    else:
        feature_defaults[col] = float(X_train_full_s[col].median())


# trained model
joblib.dump(final_model, artifact_dir / "final_model.joblib")

# preprocessing artifacts
joblib.dump({
    "sanit_map": sanit_map,
    "label_encoders": label_encoders,
    "target_transform": TARGET_TRANSFORM,
    "feature_names": feature_names,
    "feature_defaults": feature_defaults
}, artifact_dir / "preprocessing.joblib")


# save metadata

metadata = {
    "model_name": str(final_model_name),
    "target_transform": TARGET_TRANSFORM,
    "rmse_test": float(rmse_test),
    "mae_test": float(mae_test),
    "r2_test": float(r2_test),
    "median_abs_pct_error": float(np.median(abs_pct_err)),
    "n_train": int(len(X_train_full_s)),
    "n_valid": int(len(X_valid_full_s)),
    "n_test": int(len(X_test_full_s))
}

with open(artifact_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Artifacts saved to:", artifact_dir.resolve())

In [ ]:
model_card = {
    "project": "Housing Price Prediction",
    "author": "Vitalii Nechai",
    "date_created": datetime.now().strftime("%Y-%m-%d"),
    "python_version": platform.python_version(),
    "model_details": {
        "model_name": str(final_model_name),
        "target_transformation": TARGET_TRANSFORM,
        "feature_count": len(feature_names)
    },
    "data": {
        "train_size": int(len(X_train_full_s)),
        "validation_size": int(len(X_valid_full_s)),
        "test_size": int(len(X_test_full_s))
    },
    "performance_on_test": {
        "rmse": float(rmse_test),
        "mae": float(mae_test),
        "r2": float(r2_test),
        "median_absolute_percentage_error": float(np.median(abs_pct_err))
    },
    "top_features": importance_df.head(5)["feature"].tolist(),
    "limitations": [
        "Model trained on historical housing data.",
        "May not generalize to different geographic regions.",
        "Market dynamics may shift over time."
    ],
    "intended_use": "Real estate price estimation for properties similar to training distribution."
}

with open(artifact_dir / "model_card.json", "w") as f:
    json.dump(model_card, f, indent=4)

print("Model card saved.")